# 🤖 MPT Agent System — Gemini Colab Runner
**Runtime: Standard (CPU)** — Gemini API handles all LLM calls. No GPU / no local model required.

Pipeline: Trends → Script (Gemini) → Video (MPT stock footage) → SEO (Gemini) → Publish

## ⚙️ Step 1 — Install dependencies

In [ ]:
!git clone https://github.com/harry0703/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo 2>/dev/null || echo 'MPT already cloned'
!cd /content/MoneyPrinterTurbo && pip install -q -r requirements.txt
!git clone https://github.com/Bhushan-Khachane/mpt-agent-system.git /content/mpt-agent-system 2>/dev/null || (cd /content/mpt-agent-system && git pull)
!pip install -q feedparser pytrends requests google-api-python-client google-auth-httplib2 google-auth-oauthlib
print('All dependencies installed.')

## 🔑 Step 2 — Configure keys (edit here before running)

In [ ]:
import os, sys, pathlib

# ── MPT ──────────────────────────────────────────────────────────
os.environ['MPT_DIR']     = '/content/MoneyPrinterTurbo'
os.environ['MPT_USE_API'] = 'false'
os.environ['MPT_API_URL'] = 'http://127.0.0.1:8080'

# ── Gemini (only LLM key needed) ─────────────────────────────────
# Free key: https://aistudio.google.com/app/apikey
os.environ['GEMINI_API_KEY'] = 'YOUR_GEMINI_API_KEY'  # <── paste here
os.environ['GEMINI_MODEL']   = 'gemini-2.0-flash'

# ── Stock media ───────────────────────────────────────────────────
os.environ['PEXELS_API_KEY']  = 'YOUR_PEXELS_KEY'
os.environ['PIXABAY_API_KEY'] = 'YOUR_PIXABAY_KEY'

# ── YouTube / Instagram channel IDs ──────────────────────────────
os.environ['YT_CHANNEL_AI']    = 'UCxxxxxxxxxxxxxxxx'
os.environ['YT_CHANNEL_CLEAN'] = 'UCxxxxxxxxxxxxxxxx'
os.environ['YT_CHANNEL_MONEY'] = 'UCxxxxxxxxxxxxxxxx'
os.environ['IG_ACCOUNT_AI']    = 'your_ig_handle'
os.environ['IG_ACCOUNT_CLEAN'] = 'your_ig_handle'
os.environ['IG_ACCOUNT_MONEY'] = 'your_ig_handle'

sys.path.insert(0, '/content/mpt-agent-system')
for d in ['data','videos','logs','credentials']:
    pathlib.Path(f'/content/mpt-agent-system/{d}').mkdir(parents=True, exist_ok=True)
q = pathlib.Path('/content/mpt-agent-system/data/topics_queue.json')
s = pathlib.Path('/content/mpt-agent-system/data/seen_topics.json')
if not q.exists(): q.write_text('[]')
if not s.exists(): s.write_text('{}')
print('Config done. Gemini model:', os.environ['GEMINI_MODEL'])

## 📡 Step 3 — TrendScout

In [ ]:
import os
os.chdir('/content/mpt-agent-system')
from agents.trend_scout import run_trend_scout
topics = run_trend_scout()
for t in topics[:5]:
    print(f"  [{t['niche']}] {t['title']} (score={t['score_val']:.1f})")

## ✍️ Step 4 — ScriptWriter (Gemini API)

In [ ]:
from agents.script_writer import run_script_writer
run_script_writer(batch_size=6)
import json
queue    = json.loads(open('/content/mpt-agent-system/data/topics_queue.json').read())
scripted = [t for t in queue if t.get('status') == 'scripted']
if scripted:
    print(f"\n--- Sample script ({scripted[0]['niche']}) ---")
    print(scripted[0]['script'])

## 🎬 Step 5 — VideoProducer (MPT stock footage)

In [ ]:
from agents.video_producer import run_video_producer
run_video_producer(batch_size=3, use_api=False)

## 🔍 Step 6 — SEO Agent (Gemini API)

In [ ]:
from agents.seo_agent import run_seo_agent
run_seo_agent(batch_size=6)

## 🚀 Step 7 — Publisher (YouTube + Instagram + Facebook)

In [ ]:
# Place YouTube OAuth token files at:
# /content/mpt-agent-system/credentials/youtube_{niche}_token.json
from agents.publisher import run_publisher
run_publisher(batch_size=3)

## ⚡ One-shot: run full pipeline

In [ ]:
import os
os.chdir('/content/mpt-agent-system')
!python orchestrator.py